# Lab 05 — Writer Identification

> **GraphoLab** | Forensic Graphology Laboratory

**Technique:** Handcrafted feature extraction + Machine Learning classifier  
**Task:** Attribute an anonymous handwritten sample to a known writer  
**Forensic use case:** Anonymous threatening letters, disputed document authorship

---

## Approach

Writer identification uses **style features** that are consistent within a writer and discriminative across writers:

| Feature Category | Examples |
|-----------------|----------|
| **Texture** | LBP (Local Binary Patterns), GLCM, Gabor filters |
| **Gradient** | HOG (Histogram of Oriented Gradients) |
| **Run-length** | White/black run statistics along horizontal/vertical directions |
| **Structural** | Word width, letter height, ascender/descender ratios |

We extract these features from each sample, then train a **Support Vector Machine (SVM)** or **k-Nearest Neighbours (kNN)** classifier.


## GraphoLab Core — Quick Start

> The production implementation of writer identification is available in [`core/writer.py`](../core/writer.py).
> It uses a **HOG + LBP + SVM pipeline** with open-set calibration and lazy thread-safe model loading,
> and accepts a `samples_dir` parameter for flexible deployment.
>
> Run the cell below to import it directly. The remaining cells implement the same pipeline
> from scratch for educational purposes.

In [ ]:
# GraphoLab Core — production usage
# Run this cell to use the shared core module instead of the notebook implementation below.
import sys, pathlib
sys.path.insert(0, str(pathlib.Path("..").resolve()))

from core.writer import writer_identify, ensure_writer_examples
from PIL import Image
import numpy as np

# Example: identify writer of an anonymous sample
# samples_dir = pathlib.Path("../data/samples")
# doc = np.array(Image.open("../data/samples/handwritten_text_01.png").convert("RGB"))
# report, chart = writer_identify(doc, samples_dir)
# print(report)
print("core.writer imported — writer_identify() ready.")

## Setup


In [ ]:
# !pip install opencv-python scikit-learn scikit-image matplotlib numpy Pillow

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from skimage.feature import hog, local_binary_pattern
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

print("Libraries loaded.")

## Feature Extraction


In [ ]:
IMG_SIZE = (128, 256)  # height × width for normalisation


def preprocess_image(img: Image.Image) -> np.ndarray:
    """Binarise and normalise a handwriting image."""
    gray = np.array(img.convert('L'))
    # OTSU binarisation
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    # Resize
    resized = cv2.resize(binary, (IMG_SIZE[1], IMG_SIZE[0]))
    return resized


def extract_hog_features(img_binary: np.ndarray) -> np.ndarray:
    """Extract HOG (Histogram of Oriented Gradients) features."""
    features = hog(
        img_binary,
        orientations=9,
        pixels_per_cell=(16, 16),
        cells_per_block=(2, 2),
        visualize=False,
        feature_vector=True,
    )
    return features


def extract_lbp_features(img_binary: np.ndarray,
                          n_points: int = 24, radius: int = 3) -> np.ndarray:
    """Extract LBP (Local Binary Pattern) texture histogram."""
    lbp = local_binary_pattern(img_binary, n_points, radius, method='uniform')
    hist, _ = np.histogram(lbp.ravel(), bins=n_points + 2,
                            range=(0, n_points + 2), density=True)
    return hist


def extract_run_length_features(img_binary: np.ndarray) -> np.ndarray:
    """Extract horizontal and vertical run-length statistics."""
    features = []
    for axis in (1, 0):  # horizontal, vertical
        runs = []
        for line in np.apply_along_axis(lambda r: r, axis, img_binary):
            # Count consecutive white pixel runs
            changes = np.diff(np.concatenate([[0], line > 0, [0]]))
            run_starts = np.where(changes == 1)[0]
            run_ends = np.where(changes == -1)[0]
            run_lengths = run_ends - run_starts
            if len(run_lengths) > 0:
                runs.extend(run_lengths)
        arr = np.array(runs) if runs else np.array([0])
        features.extend([arr.mean(), arr.std(), arr.max()])
    return np.array(features)


def extract_features(img: Image.Image) -> np.ndarray:
    """Extract and concatenate all style features from a handwriting image."""
    binary = preprocess_image(img)
    hog_f = extract_hog_features(binary)
    lbp_f = extract_lbp_features(binary)
    run_f = extract_run_length_features(binary)
    return np.concatenate([hog_f, lbp_f, run_f])


print("Feature extraction functions ready.")
# Show feature vector size on a dummy image
dummy = Image.new('L', (256, 128), color=200)
sample_feat = extract_features(dummy)
print(f"Feature vector length: {len(sample_feat)}")

## Synthetic Dataset Generator

For demonstration without real handwriting data. In a real forensic case, replace with actual scanned samples.


In [ ]:
def make_synthetic_handwriting(writer_id: int, sample_id: int,
                                size=(256, 128)) -> Image.Image:
    """Generate a synthetic handwriting image with writer-specific style."""
    from PIL import ImageDraw
    rng = np.random.RandomState(writer_id * 100 + sample_id)

    img = Image.new('L', size, color=240)
    draw = ImageDraw.Draw(img)

    # Writer-specific style parameters
    slant = (writer_id - 3) * 5       # degrees of slant
    spacing = 18 + writer_id * 3      # letter spacing
    stroke_w = 1 + (writer_id % 3)    # stroke width
    baseline_y = 70 + rng.randint(-5, 5)

    # Draw simulated letter strokes
    x = 10
    for _ in range(rng.randint(8, 14)):
        h = rng.randint(20, 40)
        dx = int(h * np.tan(np.radians(slant)))
        draw.line([(x + dx, baseline_y - h), (x, baseline_y)],
                  fill=40, width=stroke_w)
        draw.arc([(x, baseline_y - h // 2), (x + spacing // 2, baseline_y)],
                  start=0, end=180, fill=40, width=stroke_w)
        x += spacing + rng.randint(-3, 3)
        if x > size[0] - 20:
            break

    # Add noise
    arr = np.array(img, dtype=np.float32)
    arr += rng.normal(0, 5, arr.shape)
    arr = np.clip(arr, 0, 255).astype(np.uint8)
    return Image.fromarray(arr)


def load_samples_from_dir(samples_dir: Path) -> tuple[list, list]:
    """Load writer samples from directory structure: samples_dir/writer_*/sample_*.png"""
    images, labels = [], []
    for writer_dir in sorted(samples_dir.glob("writer_*")):
        writer_id = writer_dir.name
        for img_path in sorted(writer_dir.glob("*.png")):
            images.append(Image.open(img_path))
            labels.append(writer_id)
    return images, labels


# Use real data if available, otherwise generate synthetic samples
samples_dir = Path("../data/samples")
writer_dirs = list(samples_dir.glob("writer_*"))

if writer_dirs:
    images, labels = load_samples_from_dir(samples_dir)
    print(f"Loaded {len(images)} real samples from {len(set(labels))} writers.")
else:
    N_WRITERS = 5
    N_SAMPLES_PER_WRITER = 10
    images, labels = [], []
    for w in range(N_WRITERS):
        for s in range(N_SAMPLES_PER_WRITER):
            images.append(make_synthetic_handwriting(w, s))
            labels.append(f"Writer_{w+1:02d}")
    print(f"Generated {len(images)} synthetic samples for {N_WRITERS} writers.")

# Show sample images
fig, axes = plt.subplots(1, min(5, len(set(labels))), figsize=(14, 3))
shown = set()
idx = 0
for ax in axes:
    while labels[idx] in shown:
        idx += 1
    ax.imshow(images[idx], cmap='gray')
    ax.set_title(labels[idx], fontsize=9)
    ax.axis('off')
    shown.add(labels[idx])
    idx += 1
plt.suptitle("Sample Handwriting Images (one per writer)", fontsize=11)
plt.tight_layout()
plt.show()

## Extract Features and Train Classifier


In [ ]:
print("Extracting features...")
X = np.array([extract_features(img) for img in images])

le = LabelEncoder()
y = le.fit_transform(labels)

print(f"Feature matrix: {X.shape}")
print(f"Classes: {list(le.classes_)}")

# SVM classifier pipeline
clf = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', C=10, gamma='scale', probability=True)),
])

# Cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(clf, X, y, cv=cv, scoring='accuracy')
print(f"\nCross-validation accuracy: {scores.mean():.3f} ± {scores.std():.3f}")

# Fit on full dataset for demo
clf.fit(X, y)
print("Classifier trained.")

## Demo — Identify the Author of an Anonymous Sample


In [ ]:
def identify_writer(query_image: Image.Image, top_k: int = 3) -> None:
    """Identify the most likely writer of a handwriting sample."""
    feat = extract_features(query_image).reshape(1, -1)
    probabilities = clf.predict_proba(feat)[0]

    # Ranked predictions
    ranked = sorted(zip(le.classes_, probabilities), key=lambda x: -x[1])

    fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(12, 4))

    ax_img.imshow(query_image, cmap='gray')
    ax_img.set_title("Anonymous Questioned Sample", fontsize=11)
    ax_img.axis('off')

    top = ranked[:top_k]
    names = [r[0] for r in top]
    probs = [r[1] for r in top]
    colors = ['green' if i == 0 else 'steelblue' for i in range(len(top))]
    bars = ax_bar.barh(names[::-1], probs[::-1], color=colors[::-1])
    ax_bar.set_xlim(0, 1)
    ax_bar.set_xlabel("Probability")
    ax_bar.set_title(f"Top-{top_k} Candidate Authors", fontsize=11)
    for bar, p in zip(bars, probs[::-1]):
        ax_bar.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
                    f"{p:.1%}", va='center', fontsize=10)

    plt.suptitle(f"Writer Identification — Best match: {ranked[0][0]} ({ranked[0][1]:.1%})",
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print(f"Most likely author: {ranked[0][0]} (confidence: {ranked[0][1]:.1%})")


# Use the first sample of Writer_01 as an anonymous query
anonymous_sample = images[0]  # pretend we don't know the author
identify_writer(anonymous_sample)

## Confusion Matrix (Cross-Validation)


In [ ]:
from sklearn.model_selection import cross_val_predict

y_pred = cross_val_predict(clf, X, y, cv=cv)
print(classification_report(y, y_pred, target_names=le.classes_))

fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(
    y, y_pred, display_labels=le.classes_, ax=ax, cmap='Blues'
)
ax.set_title("Writer Identification — Confusion Matrix", fontsize=12)
plt.tight_layout()
plt.show()

## Forensic Notes

- Accurate writer identification requires **at least 5–10 reference samples per writer**, covering different writing conditions (different dates, pens, speeds).
- Feature-based approaches work well for comparisons within the same language/script. Cross-language generalisation is poor.
- Deep learning approaches (CNN, ViT) outperform hand-crafted features on large datasets. See `CVxTz/handwriting_forensics` for a CNN-based approach.
- This lab provides a working baseline; for production use, collect domain-specific labelled data and fine-tune accordingly.
- Always state uncertainty explicitly in a forensic opinion — a high classifier probability is not equivalent to certainty.

---

**Next lab →** [06 — Graphological Feature Analysis](06_graphological_feature_analysis.ipynb)
